# 05 — Build and inspect a sampled PyG HeteroData

This lesson converts the deterministic fixture into a bounded heterogeneous graph using the repository's tested exporter. It inspects node maps, relation-wise edge indices, reverse-edge identity, feature coverage, and declared fallbacks. The artifact is an executable sample—not a production/full KG materialization or model-quality result.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = Path(os.environ.get("JOUVENCE_NOTEBOOK_CACHE", REPO_ROOT / "artifacts" / "cache" / "public-notebooks"))
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})

try:
    import torch
    import torch_geometric
except ImportError as exc:
    raise RuntimeError("Install the notebook GNN environment with: uv sync --group notebooks --group gnn") from exc


## Map the KG contract to PyG concepts

Each node type owns an index space and feature matrix. Each typed relation `(source_type, relation, target_type)` owns a separate edge index. Stable-ID node maps and edge-row maps must travel with tensors so positions remain auditable.


In [ ]:
pyg_contract = pd.DataFrame([
    ("nodes/<type>", "data[node_type].x", "node_id ↔ node_index"),
    ("edges/<relation>", "data[(src, rel, dst)].edge_index", "edge_key ↔ edge_pos"),
    ("reverse relation", "separate edge type", "forward_edge_pos"),
    ("features/", "x or sidecar", "coverage/fallback mask"),
], columns=["KG surface", "PyG representation", "identity sidecar"])
display(pyg_contract)


### Interpretation

A monolithic `HeteroData` pickle is appropriate only for bounded pilots. Production design is relation-wise, sidecar-first, memory-mapped, and staged to worker-local storage.


In [ ]:
print({"sample_node_types": ["gene", "disease", "molecule"], "sample_relations": ["disease_associated_gene", "molecule_targets_gene"], "bounded": True})


### Checkpoint

Before building, name the selected node types, relations, limits, and feature policy. Selection is part of the scientific artifact.


## Build the deterministic bounded export

The helper delegates to the tested exporter with strict endpoint validation, reverse edges, capped nodes and assertions, an explicit build name, and a deterministic fallback seed. Live full-KG export is refused in this laptop course.


In [ ]:
from manage_db.public_notebooks import build_sampled_pyg
if MODE != "fixture":
    raise RuntimeError("Use only an explicitly staged bounded local subset; never build the full KG on a laptop")
PYG_ROOT = CACHE / "pyg-course-sample"
result = build_sampled_pyg(KG_ROOT, PYG_ROOT, max_nodes_per_type=100, max_edges_per_relation=200)
print({"node_counts": result.node_counts, "edge_counts": result.edge_counts})


### Interpretation

A successful export proves that the selected schemas and endpoints can produce tensors. It does not prove full-scale memory behavior, training stability, useful representations, or biomedical validity.


In [ ]:
manifest = json.loads((PYG_ROOT / "manifest.json").read_text())
print(json.dumps({key: manifest.get(key) for key in ["build_name", "node_counts", "edge_counts", "include_reverse_edges"]}, indent=2))


### Checkpoint

Keep the manifest with every exported sample. Counts without source identity, limits, and feature policy are not a reproducible graph artifact.


## Load HeteroData and inspect node maps

The loader reconstructs the bounded tensor object. Node maps connect tensor positions back to biological IDs; losing them makes retrieval, evaluation, and error analysis unreliable.


In [ ]:
from manage_db.public_notebooks import load_sampled_pyg
data = load_sampled_pyg(PYG_ROOT)
print(data)
print("node types:", data.node_types)


### Interpretation

Node index zero in one type is unrelated to index zero in another type. Every relation edge index is interpreted in its source and target type-specific spaces.


In [ ]:
for node_type in data.node_types:
    store = data[node_type]
    print(node_type, {"nodes": store.num_nodes, "x_shape": tuple(store.x.shape), "finite": bool(store.x.isfinite().all())})


### Checkpoint

Verify node counts, feature dimensions, and finite values before running any model. Then use persisted maps—not DataFrame row assumptions—to recover stable IDs.


## Inspect relation-wise edge indices and reverse identity

Each relation has a `2 × E` integer edge index. Reverse edges support bidirectional message passing but reuse forward biological identity; they do not create new evidence or a second biological assertion.


In [ ]:
for edge_type in data.edge_types:
    edge_index = data[edge_type].edge_index
    print(edge_type, {"shape": tuple(edge_index.shape), "min": int(edge_index.min()), "max": int(edge_index.max())})


### Interpretation

The same relation name can only be interpreted with its source and target types. Bounds must be checked against the corresponding node counts, not a global node total.


In [ ]:
for src, relation, dst in data.edge_types:
    edge_index = data[(src, relation, dst)].edge_index
    assert edge_index.shape[0] == 2
    assert int(edge_index[0].max()) < data[src].num_nodes
    assert int(edge_index[1].max()) < data[dst].num_nodes
print("all sampled edge indices respect typed node bounds")


### Checkpoint

A reverse relation is a computational view. Do not attach independent source provenance or count it as additional canonical topology.


## Audit feature coverage and declared fallbacks

Each node type should distinguish source-backed available, absent, deferred, and fallback states. Joined source vectors remain source-backed only for covered rows; a deterministic or learned fallback enables execution but is not biological evidence.


In [ ]:
metadata_path = PYG_ROOT / "heterodata" / "full_graph.metadata.json"
export_metadata = json.loads(metadata_path.read_text())
feature_policy = export_metadata.get("node_embedding_policy", {})
print(json.dumps(feature_policy, indent=2))


### Interpretation

Availability does not mean complete coverage. Zero-filling can hide absence and change geometry, so missing rows need an explicit coverage or fallback mask and a manifest policy.


In [ ]:
coverage_rows = []
for node_type in data.node_types:
    store = data[node_type]
    coverage_rows.append({"node_type": node_type, "rows": store.num_nodes, "feature_dim": store.x.shape[1], "all_finite": bool(store.x.isfinite().all())})
display(pd.DataFrame(coverage_rows))


### Checkpoint

When adapting this sample, report source-backed row coverage separately from fallback row coverage. Never describe the fallback seed as a foundation embedding.


## Run structural invariants and state the boundary

A trustworthy sample has nonempty selected stores, finite features, in-range edge indices, deterministic counts, and manifests/maps that remain beside tensors. These checks establish software integrity for this bounded artifact.


In [ ]:
assert set(data.node_types) == {"gene", "disease", "molecule"}
assert all(data[node_type].num_nodes > 0 for node_type in data.node_types)
assert all(data[edge_type].edge_index.shape[1] > 0 for edge_type in data.edge_types)
print("bounded HeteroData structural checks: PASS")


### Interpretation

What this means: Jouvence's typed fixture can be exported and loaded through the real PyG pipeline. What this does not mean: a full production export exists, all node features are source-backed, training is stable, or a prediction is valid.


In [ ]:
print({
    "artifact": str(PYG_ROOT),
    "status": "bounded exporter smoke passed",
    "production_full_done": False,
    "next_notebook": "06_sampled_ml_use_cases.ipynb",
})


### Checkpoint

Proceed to Notebook 06 for retrieval, neighborhood inspection, split design, negative sampling, metrics, and error analysis on this same honest smoke boundary.
